# Gold Layer Aggregation Pipeline - Generic & Metadata-Driven

Generic notebook to combine, join, and aggregate Silver tables into Gold business marts based on config.

In [ ]:
# Notebook Parameters
dbutils.widgets.text("config_dir", "/Workspace/Users/jayarampogakula@gmail.com/lakeforge/configs", "Config Directory")
dbutils.widgets.text("business_domain", "sales", "Business Domain")
dbutils.widgets.text("target_table", "customer_summary", "Target Gold Table")
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"], "Environment")

config_dir = dbutils.widgets.get("config_dir")
business_domain = dbutils.widgets.get("business_domain")
target_table = dbutils.widgets.get("target_table")
environment = dbutils.widgets.get("environment")

print(f"Executing Gold Aggregation for {target_table} ({business_domain} domain, {environment} environment)")

In [ ]:
# Imports & Path setup
import sys
sys.path.append("/Workspace/Users/jayarampogakula@gmail.com/lakeforge")

from lakeforge import (
    ConfigParser,
    GoldAggregator,
    create_trust_engine
)

print("✅ Framework libraries loaded")

In [ ]:
# Load Configurations
env_config = ConfigParser.get_environment_config(config_dir, environment)

catalog = env_config["catalog"]
silver_schema = env_config["silver_schema"]
gold_schema = env_config["gold_schema"]

gold_config = ConfigParser.get_gold_aggregation_config(config_dir, target_table, environment)
print(f"Parsed Gold Config for: {target_table}")
print(gold_config)

In [ ]:
# Load Silver source tables dynamically
silver_dfs = {}
for alias, table_path in gold_config["source_tables"].items():
    # Extract basename, e.g., 'silver.customers_clean' -> 'customers_clean'
    table_basename = table_path.split(".")[-1]
    full_silver_path = f"{catalog}.{silver_schema}.{table_basename}"
    
    print(f"Loading source: {alias} -> {full_silver_path}")
    silver_dfs[alias] = spark.table(full_silver_path)

print(f"✅ Loaded {len(silver_dfs)} source Silver dataframes")

In [ ]:
# Execute aggregations dynamically
result_df = GoldAggregator.aggregate(spark, silver_dfs, gold_config)
print(f"✅ Aggregation complete. Columns: {result_df.columns}")

In [ ]:
# Write / Merge to Gold layer
merge_strat = gold_config.get("merge_strategy", "overwrite")
business_keys = gold_config.get("business_key", [])

full_gold_table = f"{catalog}.{gold_schema}.{target_table}"

if merge_strat in ["upsert", "merge"] and business_keys:
    print(f"ℹ️ Performing merge/upsert on Gold table matching keys {business_keys}...")
    from lakeforge.silver.merge_engine import SilverMergeEngine
    merge_stats = SilverMergeEngine.merge(
        spark=spark,
        df=result_df,
        target_table=full_gold_table,
        merge_keys=business_keys,
        mode="merge"
    )
    print(f"✅ Merge complete: {merge_stats}")
elif merge_strat == "append":
    print("ℹ️ Performing append on Gold table...")
    from lakeforge.silver.merge_engine import SilverMergeEngine
    merge_stats = SilverMergeEngine.merge(
        spark=spark,
        df=result_df,
        target_table=full_gold_table,
        merge_keys=[],
        mode="append"
    )
    print(f"✅ Append complete: {merge_stats}")
else:
    print("ℹ️ Performing complete overwrite on Gold table...")
    result_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(full_gold_table)
    print(f"✅ Overwrite complete")

In [ ]:
# Compute and Log Trust Score
trust_engine = create_trust_engine(spark)

# Compare first source table row count with target aggregated row count for reconciliation
first_source = list(silver_dfs.values())[0]
trust_validations = [
    {
        "type": "row_count",
        "params": {
            "source_df": first_source,
            "target_df": result_df,
            "tolerance_percent": 100.0  # Gold aggregations naturally compress row counts, so tolerance is high
        }
    }
]

trust_results = trust_engine.run_trust_validations(trust_validations)
trust_score = trust_engine.calculate_trust_score(
    table_name=full_gold_table,
    dq_results=trust_results,
    pipeline_stage="gold"
)

print(f"🎯 Pipeline Trust Score: {trust_score['overall_score']:.1f}%")
print(f"🎯 Trust Level: {trust_score['trust_level']}")

In [ ]:
# Create Views if configured
try:
    print("Checking for configured views to create...")
    from lakeforge.views.view_manager import create_view_manager
    from lakeforge.metadata.config_parser import load_view_configs
    
    # Load env config for query formatting
    env_config = ConfigParser.get_environment_config(config_dir, environment)
    
    # Load all views
    view_configs = load_view_configs(config_dir, environment)
    
    # Filter views that belong to the current catalog/schema
    target_views = [vc for vc in view_configs if vc.schema == env_config["gold_schema"]]
    
    if target_views:
        view_manager = create_view_manager(spark, env_config)
        view_manager.create_views(target_views)
        print(f"✅ Created {len(target_views)} views successfully")
    else:
        print("No views configured for this schema/layer")
except Exception as e:
    print(f"⚠️ Error creating views: {str(e)}")